# C6 on Colab L4 — regex + Kimi + **Grok** soft labels

**The experiment.** Identical to C2 in every respect except the label source:

| | C2 (LB 0.913) | C3 (58-gold 0.9094) | **C6 (this run)** |
|---|---|---|---|
| corpus | public 44x336 memmap | same | same |
| backbone / MIL | CoAtNet-384, k=8/24, bs=2, 9 epochs | same | same |
| soft labels | regex + Kimi | regex + Kimi + Qwen | regex + Kimi + **Grok** |
| 3rd source vs Kimi | — | r = **0.94** (redundant) | r = **0.76** (decorrelated) |

C3 added a redundant third source and moved the 58-gold held-out by **+0.0013** — nothing.
C6 repeats that exact structural change with a *decorrelated* third source. If the
independence hypothesis is right, C6 > C3. If C6 also sits on C2, the whole
"add another labeller" direction is dead and the label work has to change shape.

Offline GAUGE 1 does **not** back C6 (0.8931 vs C3's 0.8941) — but every gap in that
table is smaller than Kimi's own 0.018 run-to-run scatter on 58 studies, and C3 already
showed the gauge extrapolates badly. The run is the adjudicator.

---

**Why Colab and not Kaggle:** this is ~6h of Kaggle GPU quota that we do not have this
week. Training here costs none of it; only the final inference kernel does.

**Before you start:** set Runtime → Change runtime type → **L4 GPU**, and have a Kaggle
**access token** ready (`kaggle auth print-access-token --expiration 6h` on your own
machine). This account does not use the legacy `kaggle.json`. Expect ~20 min of setup
and ~2.5-3h of training.


In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), 'no GPU -- Runtime > Change runtime type > L4 GPU'
cap = torch.cuda.get_device_capability()
print(torch.cuda.get_device_name(0), '| capability', cap,
      '| VRAM %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024**3))
print('AMP dtype corpus.py will pick:', 'bfloat16' if cap[0] >= 8 else 'float16')
# L4 is sm_89 -> bf16 -> the GradScaler is DISABLED, so there is no built-in
# inf/nan step skip. corpus.py carries its own guard (added after the R9700 run
# went loss=nan at epoch 5 and never recovered); watch for 'skip=N' in the epoch lines.
print(subprocess.run(['df', '-h', '/content'], capture_output=True, text=True).stdout)
# The corpus is 21.9 GB and must live on /content (local SSD).
# NEVER put it on Drive: memmap does random reads and Drive FUSE would crawl.


## 1. Dependencies

`kaggle>=2.2` is **not optional**: Colab preinstalls kaggle 1.x, which only understands
the legacy `kaggle.json`. This account authenticates with an *access token*, and only
2.2+ reads one.


In [ ]:
!pip -q install -U 'kaggle>=2.2' timm==1.0.29 pydicom pyarrow 2>&1 | tail -2
import timm, subprocess
ver = subprocess.run(['kaggle', '--version'], capture_output=True, text=True).stdout.strip()
print('timm', timm.__version__, '|', ver)
assert tuple(int(x) for x in ver.split()[-1].split('.')[:2]) >= (2, 2), \
    'kaggle CLI too old to read an access token -- restart the runtime after the pip upgrade'


## 2. Kaggle credentials — access token, **not** `kaggle.json`

The CLI resolves credentials in this order (`kagglesdk/kaggle_env.py`):
`KAGGLE_API_TOKEN` env → `~/.kaggle/access_token` → legacy `kaggle.json` → OAuth.
This account has an access token, so we set the first two.

**Get one** (either works):
- on your own machine: `kaggle auth print-access-token --expiration 6h`
- or <https://www.kaggle.com/settings/api> → *Generate New Token*

`getpass` keeps the token out of the cell output, so the saved notebook carries no secret.

**Tokens expire.** That is fine here: Kaggle is only touched during the ~20 min of
downloads in step 3. Training reads from local disk and never calls the API, so a token
that lapses mid-run costs nothing. Ask for `--expiration 6h` and stop worrying about it.


In [ ]:
import os, getpass, subprocess

tok = getpass.getpass('Kaggle access token (input hidden): ').strip()
assert tok and '{' not in tok, 'that looks like kaggle.json contents, not an access token'
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(tok)
os.chmod('/root/.kaggle/access_token', 0o600)
os.environ['KAGGLE_API_TOKEN'] = tok      # also covers the '!' subshells below
del tok

# whoami, without printing anything sensitive
r = subprocess.run(['kaggle', 'datasets', 'list', '-m', '-s', 'rsna-knee-src'],
                   capture_output=True, text=True)
print(r.stdout[:400] or r.stderr[:400])
assert 'rsna-knee-src' in r.stdout, 'auth failed -- token expired or wrong account'
print('auth OK')


## 3. Fetch everything from Kaggle

- `easonyanyan/rsna-knee-src` — the project's `src/` (config.py already set to C6)
- `easonyanyan/rsna-knee-llm-soft-labels` — kimi / qwen / grok parquets
- `easonyanyan/timm-coatnet-rmlp2-384` — offline CoAtNet weights
- `dreaddevelopment/knee-raptor-corpus` + `-ext` — the 44-slice memmap (21.9 GB)
- competition `train.csv` / `train_series.csv`


In [ ]:
ROOT = '/content/rsna-knee'
!mkdir -p {ROOT}/src {ROOT}/data/raw {ROOT}/data/labels {ROOT}/data/timm_weights {ROOT}/data/corpus44 {ROOT}/experiments

!rm -rf /tmp/dl
!kaggle datasets download -d easonyanyan/rsna-knee-src -p /tmp/dl --unzip -q
# verify BEFORE destroying the old copy: a failed download (expired token) must
# not leave the session with no src/ at all.
import os as _os
assert _os.path.exists('/tmp/dl/config.py') and _os.path.isdir('/tmp/dl/backbones'), (
    'src download failed -- token expired? re-run the step-2 auth cell')
!rm -rf {ROOT}/src && mkdir -p {ROOT}/src && cp -r /tmp/dl/. {ROOT}/src/
# -r, not *.py: src/ contains the `backbones` PACKAGE, which model.py imports
# (train.py -> model.py -> backbones). Copying only *.py silently drops it and
# the run dies at ModuleNotFoundError minutes in, not here.
!kaggle datasets download -d easonyanyan/rsna-knee-llm-soft-labels -p {ROOT}/data/labels --unzip -q
!kaggle datasets download -d easonyanyan/timm-coatnet-rmlp2-384 -p {ROOT}/data/timm_weights --unzip -q
!ls {ROOT}/src | head -20; ls -la {ROOT}/data/labels {ROOT}/data/timm_weights


In [ ]:
# competition CSVs (accept the competition rules on Kaggle first, or this 403s)
!kaggle competitions download -c rsna-knee-abnormality-detection -f train.csv -p {ROOT}/data/raw -q
!kaggle competitions download -c rsna-knee-abnormality-detection -f train_series.csv -p {ROOT}/data/raw -q
!cd {ROOT}/data/raw && ls *.zip >/dev/null 2>&1 && (unzip -oq '*.zip'; rm -f *.zip) || true
!ls -la {ROOT}/data/raw


In [ ]:
# 21.9 GB corpus -> /content local disk. Takes ~10-20 min.
!kaggle datasets download -d dreaddevelopment/knee-raptor-corpus -p {ROOT}/data/corpus44 --unzip -q
!kaggle datasets download -d dreaddevelopment/knee-raptor-corpus-ext -p {ROOT}/data/corpus44 --unzip -q
!ls -la {ROOT}/data/corpus44; df -h /content


## 4. Pre-flight — verify the experiment is what we think it is
Cheap assertions that would otherwise only surface 40 minutes into training.


In [ ]:
import os, sys
os.environ['RSNA_DATA_ROOT']     = f'{ROOT}/data/raw'
os.environ['RSNA_CORPUS_DIR']    = f'{ROOT}/data/corpus44'
os.environ['RSNA_CORPUS_SLOTS']  = '44'      # published memmap, NOT our 64-slice shards
os.environ['RSNA_OUT_DIR']       = f'{ROOT}/experiments/c6_44_kimi_grok'
os.environ['RSNA_TIME_BUDGET_H'] = '9'       # stop cleanly before a session cap
sys.path.insert(0, f'{ROOT}/src')

from config import (SOFT_LABEL_PARQUETS, CORPUS_K_TRAIN, CORPUS_K_EVAL,
                    CORPUS_EPOCHS, CORPUS_BATCH)
print('label sources :', SOFT_LABEL_PARQUETS)
print('k / ep / bs   :', CORPUS_K_TRAIN, CORPUS_K_EVAL, CORPUS_EPOCHS, CORPUS_BATCH)
assert SOFT_LABEL_PARQUETS == ['labels_kimi_soft.parquet',
                               'labels_grok_soft.parquet'], SOFT_LABEL_PARQUETS
assert (CORPUS_K_TRAIN, CORPUS_K_EVAL, CORPUS_EPOCHS, CORPUS_BATCH) == (8, 24, 9, 2), \
    'not the C2 budget -- C6 would differ from C2 in more than the label source'

# corpus.py pulls these lazily, mid-run (corpus.py:488-489 and :675). Import them
# NOW so a missing module fails in 2 seconds instead of after the corpus is open:
# `train` -> `model` -> `backbones` is how the DINOv2 line's package gets dragged
# into the C line, and it is exactly what a *.py-only copy of src/ leaves out.
from raptor import RaptorClassifier, build_raptor_backbone, _recalibrate_bn
from train import macro_auc, _write_submission
print('lazy imports  : OK')

import corpus
print('CORPUS_D      :', corpus.CORPUS_D)
assert corpus.CORPUS_D == 44
vols, masks, ids = corpus.open_corpus()
assert vols.shape == (4407, 44, 336, 336), vols.shape
del vols, masks, ids

from dataset import RSNADataset
ds = RSNADataset('train')
print('targets mean %.3f | weights %.3f-%.3f'
      % (ds.targets.mean(), ds.weights.min(), ds.weights.max()))
# expect: both parquets hit, '4407/4407 studies got LLM labels (0 extractor-only)',
#         targets mean ~0.32, weights 0.287-3.0


## 5. Train
Launched detached so a browser disconnect does not kill it. The cell after this one
tails the log — re-run it any time to catch up.


In [ ]:
import time
LOG = f'{ROOT}/experiments/c6_44_kimi_grok/train.log'
!mkdir -p {ROOT}/experiments/c6_44_kimi_grok
!cd {ROOT} && RSNA_DATA_ROOT={ROOT}/data/raw RSNA_CORPUS_DIR={ROOT}/data/corpus44 RSNA_CORPUS_SLOTS=44 RSNA_OUT_DIR={ROOT}/experiments/c6_44_kimi_grok RSNA_TIME_BUDGET_H=9 PYTHONPATH={ROOT}/src nohup python -u src/train_local.py > {LOG} 2>&1 &
time.sleep(45)
!tail -20 {LOG}


In [ ]:
!tail -30 {ROOT}/experiments/c6_44_kimi_grok/train.log
!nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv
# Watch for 'amp dtype: torch.bfloat16' and the per-epoch 'gold=' column.
# A 'skip=N' suffix means the NaN guard caught bad steps: a handful is fine,
# hundreds means the run is unstable and the LR needs looking at.


## 6a. Mid-training safety copy → Drive

Colab's disk is ephemeral: whatever is not copied out dies with the runtime.
`corpus0.pt` is rewritten (atomically) every time the gold score improves, so this cell
is safe to run **during** training and worth running more than once.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EXP = f'{ROOT}/experiments/c6_44_kimi_grok'
!mkdir -p /content/drive/MyDrive/rsna-knee
!cp {EXP}/corpus0.pt /content/drive/MyDrive/rsna-knee/c6_corpus0.pt
!cp {EXP}/train.log  /content/drive/MyDrive/rsna-knee/c6_train.log
!ls -la /content/drive/MyDrive/rsna-knee


## 6b. When training is DONE → publish the checkpoint as a Kaggle dataset

This is the route that actually matters, and it does double duty:

1. it is how the checkpoint gets **off Colab and down to the local machine**
   (`kaggle datasets download` there — no 279 MB browser download, no Drive juggling), and
2. the LB inference kernel has to **mount** the weights anyway, and kernels mount datasets.

Run it only after `corpus run finished in ... min` appears in the log — a dataset version
is heavyweight, unlike the Drive copy above.

> If this 401s, the access token has expired (they are short-lived). Re-run the step-2
> cell with a fresh one; the training output on disk is untouched.


In [ ]:
import os, json, shutil, subprocess, torch
EXP = f'{ROOT}/experiments/c6_44_kimi_grok'

ck = torch.load(f'{EXP}/corpus0.pt', map_location='cpu', weights_only=False)
print('gold_auc :', ck['gold_auc'])
print('res      :', ck['res'], '| slots', ck['slots'])
assert 'epoch' not in ck, 'this is a MID-RUN snapshot -- training has not finished yet'

UP = '/content/upload_c6'
os.makedirs(UP, exist_ok=True)
shutil.copy(f'{EXP}/corpus0.pt', f'{UP}/c6_corpus0.pt')
shutil.copy(f'{EXP}/train.log',  f'{UP}/c6_train.log')
json.dump({'title': 'RSNA Knee C6 weights (regex+kimi+grok, 44-slice)',
           'id': 'easonyanyan/rsna-knee-c6-weights',
           'licenses': [{'name': 'CC0-1.0'}]},
          open(f'{UP}/dataset-metadata.json', 'w'))

msg = f"C6 44-slice regex+kimi+grok, gold {ck['gold_auc']:.4f}"
# 'create' vs 'version' decided by whether the dataset already exists -- do not
# string-match on an error message for this.
exists = 'rsna-knee-c6-weights' in subprocess.run(
    ['kaggle', 'datasets', 'list', '-m', '-s', 'rsna-knee-c6-weights'],
    capture_output=True, text=True).stdout
cmd = (['kaggle', 'datasets', 'version', '-p', UP, '-m', msg, '-r', 'skip'] if exists
       else ['kaggle', 'datasets', 'create', '-p', UP, '-r', 'skip'])
print('running:', ' '.join(cmd[:4]))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-800:] or r.stderr[-800:])
assert r.returncode == 0, 'upload failed -- token expired? re-run the step-2 cell'


## 6c. (optional) Unattended: wait → publish → shut the runtime down

**Nothing above stops the Colab session.** When training exits, the VM stays up and keeps
burning compute units until you stop it or Colab times it out. If you are going to walk
away, run this cell *instead of* 6a/6b: it blocks until the run finishes, publishes the
dataset, and only then calls `runtime.unassign()`.

The ordering is deliberate — **it never shuts down on a failed upload.** If the token has
expired the cell stops loudly and leaves the VM alive so the weights are still recoverable.

If your browser disconnects and this watcher dies, nothing is lost: the training itself is
`nohup`-ed and unaffected, and you can just run 6b by hand afterwards.


In [ ]:
import os, json, shutil, subprocess, time, torch
EXP  = f'{ROOT}/experiments/c6_44_kimi_grok'
LOG  = f'{EXP}/train.log'
DONE = 'corpus run finished in'

while True:
    log = open(LOG).read()
    alive = subprocess.run(['pgrep', '-f', 'train_local.py'],
                           capture_output=True).returncode == 0
    if DONE in log:
        break
    if not alive:
        raise SystemExit('training exited WITHOUT finishing -- last lines:\n'
                         + '\n'.join(log.splitlines()[-15:]))
    print(log.splitlines()[-1][:110], flush=True)
    time.sleep(300)

print('\n'.join(open(LOG).read().splitlines()[-14:]))

ck = torch.load(f'{EXP}/corpus0.pt', map_location='cpu', weights_only=False)
assert 'epoch' not in ck, 'still a mid-run snapshot'
UP = '/content/upload_c6'
os.makedirs(UP, exist_ok=True)
shutil.copy(f'{EXP}/corpus0.pt', f'{UP}/c6_corpus0.pt')
shutil.copy(LOG, f'{UP}/c6_train.log')
json.dump({'title': 'RSNA Knee C6 weights (regex+kimi+grok, 44-slice)',
           'id': 'easonyanyan/rsna-knee-c6-weights',
           'licenses': [{'name': 'CC0-1.0'}]},
          open(f'{UP}/dataset-metadata.json', 'w'))
msg = f"C6 44-slice regex+kimi+grok, gold {ck['gold_auc']:.4f}"
exists = 'rsna-knee-c6-weights' in subprocess.run(
    ['kaggle', 'datasets', 'list', '-m', '-s', 'rsna-knee-c6-weights'],
    capture_output=True, text=True).stdout
cmd = (['kaggle', 'datasets', 'version', '-p', UP, '-m', msg, '-r', 'skip'] if exists
       else ['kaggle', 'datasets', 'create', '-p', UP, '-r', 'skip'])
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-600:] or r.stderr[-600:])

# ONLY now is it safe to give up the machine.
if r.returncode != 0:
    raise SystemExit('UPLOAD FAILED -- runtime left alive on purpose. '
                     'Re-run the step-2 auth cell, then cell 6b.')
print(f"published. gold_auc = {ck['gold_auc']:.4f}. shutting the runtime down.")
from google.colab import runtime
runtime.unassign()


## 7. Pull it to the local machine

On the workstation (the Kaggle CLI there already has a valid access token):

```bash
kaggle datasets download -d easonyanyan/rsna-knee-c6-weights \
  -p "D:/RSNA Knee Abnormality Detection/experiments/c6_44_kimi_grok" --unzip
```

Then verify it survived the trip, and that its geometry matches what the inference
kernel will build:

```python
import torch
ck = torch.load('experiments/c6_44_kimi_grok/c6_corpus0.pt', map_location='cpu',
                weights_only=False)
print(ck['gold_auc'], ck['res'], len(ck['slots']), sum(s[2] for s in ck['slots']))
# expect: <best gold>  384  5  44
```


## 9. Calibration arm — C2-repro on **this same L4**

C6 came in at 0.8927 against C2's 0.9081, but C1–C3 all ran on Kaggle **T4 / fp16**
while C6 ran here on **L4 / bf16**. Hardware and AMP dtype moved together with the
label source, so that −0.0154 cannot be attributed yet.

This run pins it down: same machine, same everything, label source rolled back to
`regex + kimi`. Land near **0.892** → the gap is hardware/dtype and Grok is neutral.
Land near **0.908** → Grok genuinely hurt.

It is worth 3.3h for a second reason: **C5 (R9700 / bf16) and every future off-Kaggle
arm need this same T4↔bf16 offset to be comparable to C1–C3 at all.**

The label list now comes from `RSNA_SOFT_LABELS`, so treatment and control run the
*identical* published `src/` — a control that differs from its treatment arm by a
re-upload is not a control.


In [ ]:
import time
OUT2 = f'{ROOT}/experiments/c2repro_44_kimi'
LOG2 = f'{OUT2}/train.log'
!rm -rf /tmp/dl
!kaggle datasets download -d easonyanyan/rsna-knee-src -p /tmp/dl --unzip -q
import os as _os
assert _os.path.exists('/tmp/dl/config.py') and _os.path.isdir('/tmp/dl/backbones'), (
    'src download failed -- token expired? re-run the step-2 auth cell')
!rm -rf {ROOT}/src && mkdir -p {ROOT}/src {OUT2} && cp -r /tmp/dl/. {ROOT}/src/
!cd {ROOT} && RSNA_DATA_ROOT={ROOT}/data/raw RSNA_CORPUS_DIR={ROOT}/data/corpus44 RSNA_CORPUS_SLOTS=44 RSNA_SOFT_LABELS=labels_kimi_soft.parquet RSNA_OUT_DIR={OUT2} RSNA_TIME_BUDGET_H=9 PYTHONPATH={ROOT}/src nohup python -u src/train_local.py > {LOG2} 2>&1 &
time.sleep(45)
!tail -20 {LOG2}
# MUST read: label sources: regex + ['labels_kimi_soft.parquet']
# and k 8/24 | bs 2 | ep 9 -- identical to C6 in everything but the label list.


## 8. What to report back

- the per-epoch `gold=` column and the best value
- any `skip=` counts (the NaN guard; a handful per epoch is normal on bf16)
- total wall-clock

**The number that decides C6: best 58-gold held-out, against C2 = 0.9081 and C3 = 0.9094.**

Scoring it on the leaderboard is a separate, cheap step — an inference-only Kaggle kernel
that mounts `easonyanyan/rsna-knee-c6-weights`, not another 6h training kernel.
